# Notebook to get started NN monitoring

In [1]:
from dataset import Dataset
from feature_extractor import FeatureExtractor
# from monitors import *
from evaluation import Evaluator

import torch

In [2]:
batch_size = 100
device_name = 'cuda:0' if torch.cuda.is_available() else 'cpu'

## Parameters definition

In this section, we define the parameters of the benchmark study :
* The in-distribution dataset
* The out-distribution dataset
* The out-distribution adv attack is one is used
* The network model used
* The network layer index to monitor

In [3]:
# Define the ID and OOD datasets
id_dataset = "cifar10"
ood_dataset = "cifar10"

# Define the model and layer to monitor
model = "densenet"
layer = 98

# Define the adv attack to generate the OOD dataset
atk = "fgsm"

## Create the `Dataset` objects

In this section, we use the `Dataset` object to create datasets containing the train, the test, and the OOD samples.

In [4]:
dataset_train = Dataset(id_dataset, "train", model, batch_size=batch_size)
dataset_test = Dataset(id_dataset, "test", model, batch_size=batch_size)
dataset_ood = Dataset(ood_dataset, "test", model, None, atk, batch_size=batch_size)

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


## Create the `FeatureExtractor` object

In [5]:
feature_extractor = FeatureExtractor(model, id_dataset, [layer], device_name)

Je commence l'extraction


c:\Users\MathieuDARIO\Workspace\neural-network-monitoring-benchmark\feature_extractor.py:315: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.

J'ai fini


In [6]:
features_train, logits_train, softmax_train, \
    pred_train, lab_train = feature_extractor.get_features(dataset_train)
features_test, logits_test, softmax_test, \
    pred_test, lab_test = feature_extractor.get_features(dataset_test)
features_ood, logits_ood, softmax_ood, \
    pred_ood, lab_ood = feature_extractor.get_features(dataset_ood)

In [7]:
print(features_train[0].shape)
print(logits_train.shape)
print(softmax_train.shape)
print(pred_train.shape)
print(lab_train.shape)

(50000, 342)
(50000, 10)
(50000, 10)
(50000,)
(50000,)


## Create the `Evaluator` object

In [8]:
eval_oms = Evaluator("oms", is_novelty=(id_dataset != ood_dataset))
eval_oms.fit_ground_truth(lab_test, lab_ood, pred_test, pred_ood)

## Create the monitors

In [11]:
from Monitors import *

### The Max-softmax Probability monitor

In [9]:
# monitor = MaxSoftmaxProbabilityMonitor()

# scores_test = monitor.predict(softmax_test)
# scores_ood  = monitor.predict(softmax_ood)

### The Max-logit monitor

In [10]:
# monitor = MaxLogitMonitor()

# scores_test = monitor.predict(logits_test)
# scores_ood  = monitor.predict(logits_ood)

### The Energy monitor

In [11]:
# monitor = EnergyMonitor()

# scores_test = monitor.predict(logits_test)
# scores_ood  = monitor.predict(logits_ood)

### The Outside-the-box monitor

In [12]:
monitor = OutsideTheBoxMonitor(id_dataset, model, layer, n_clusters=20)

monitor.fit(features_train[0], pred_train, lab_train, save=True)
scores_test = monitor.predict(features_test[0], pred_test)
scores_ood = monitor.predict(features_ood[0], pred_ood)

In [13]:
# Get the different metrics
aupr = eval_oms.get_average_precision(scores_test, scores_ood)
auroc = eval_oms.get_auroc(scores_test, scores_ood)
tnr95tpr = eval_oms.get_tnr_frac_tpr_oms(scores_test, scores_ood, frac=.95)

In [15]:
monitor = GaussianMixtureMonitor(id_dataset, model, layer, n_components=5,
                                    constraint="diag", is_cv=True)
monitor.fit(features_train[0], pred_train, lab_train, save=True)

scores_test = monitor.predict(features_test[0], pred_test)
scores_ood = monitor.predict(features_ood[0], pred_ood)

aupr = eval_oms.get_average_precision(-scores_test, -scores_ood)
auroc = eval_oms.get_auroc(-scores_test, -scores_ood)
tnr95tpr = eval_oms.get_tnr_frac_tpr_oms(-scores_test, -scores_ood, frac=0.95)

c:\Users\MathieuDARIO\Logiciels\miniconda3\envs\nn-monitoring-benchmark\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator GaussianMixture from version 1.1.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [14]:
print(aupr, auroc, tnr95tpr)

0.24477102689445387 0.4596115064747043 0.06080894533406955
